# Filtrado colaborativ ítem - ítem basado en KNN

El filtrado colaborativo ítem-ítem basado en KNN es una técnica de recomendación que sugiere a un usuario elementos similares a los que ya ha evaluado positivamente en el pasado. En lugar de buscar usuario con gustos parecidos, analiza las coincidencias en los patrones de calificación que recibe cada objeto por parte de toda la comunidad.

Se toma un muestra de 10.000 ítems (populares) para reducir el uso de memoria y garantizar que el entrenamiento de K-NN sea computacionalmente viable en este entorno.

In [1]:
import pandas as pd

In [2]:
ratings_df = pd.read_csv('ratings_limpios.csv')
resumen_usuario = pd.read_csv('resumen_usuario.csv')

In [5]:
items_frecuentes = (
    ratings_df["ISBN"]
    .value_counts()
    .head(10000)
    .index
    )

ratings_item = ratings_df[
    ratings_df["ISBN"].isin(items_frecuentes)
    ].copy()


In [6]:
from surprise import Dataset, Reader, KNNWithMeans
from surprise.model_selection import train_test_split

reader = Reader(rating_scale=(1,10))
surprise_df = ratings_item.copy()
surprise_df.columns = ["uid", "iid", "rating"]

data = Dataset.load_from_df(surprise_df, reader)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

algo = KNNWithMeans(k=40, sim_options={'name': 'cosine', 'user_based': False})
algo.fit(trainset)
predictions = algo.test(testset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [7]:
# Transformación a dataframe
preds_df = pd.DataFrame([
    {
        'User-ID':pred.uid,
        'ISBN': pred.iid,
        'r_ui': pred.r_ui,
        'est': pred.est
    }
    for pred in predictions
])

In [8]:
from metrics import evaluar_metricas_usuario

K = 10
metricas_usuarios = (
    preds_df
    .groupby('User-ID')
    .apply(evaluar_metricas_usuario, k=K)
    .reset_index()
)


df_evaluacion = pd.merge(
    metricas_usuarios,
    resumen_usuario,
    on='User-ID',
    how='inner',
    validate='one_to_one'
)

In [9]:
# A. Por Grupo Etario (Demográfico)
print(f"=== Métricas por Grupo Etario (K={K}) ===")
print(df_evaluacion.groupby('Grupo_Etario')[['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']].mean())

# B. Por Historial de Interacciones (Comportamiento)
print(f"\n=== Métricas por Historial de Interacciones (0: Corto, 1: Largo) (K={K}) ===")
print(df_evaluacion.groupby('Grupo_Historial')[['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']].mean())

# C. Por Grado de Exigencia (Comportamiento)
print(f"\n=== Métricas por Grado de Exigencia (0: Exigente, 1: Normal, 2: Generoso) (K={K}) ===")
print(df_evaluacion.groupby('Grupo_Exigencia')[['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']].mean())

=== Métricas por Grupo Etario (K=10) ===
                   MAE      CG@10     DCG@10   NDCG@10
Grupo_Etario                                          
Adultos       1.364386  16.817699  12.113933  0.988574
Jóvenes       1.353126  14.232877  11.141821  0.991524
Mayores       1.322503  11.678344   9.732457  0.992880

=== Métricas por Historial de Interacciones (0: Corto, 1: Largo) (K=10) ===
                      MAE      CG@10     DCG@10   NDCG@10
Grupo_Historial                                          
0                1.355385   7.659243   7.648755  0.999949
1                1.387723  17.292665  12.393151  0.986991

=== Métricas por Grado de Exigencia (0: Exigente, 1: Normal, 2: Generoso) (K=10) ===
                      MAE      CG@10     DCG@10   NDCG@10
Grupo_Exigencia                                          
0                2.561061   7.273732   6.199204  0.992666
1                1.207689  15.626833  11.602232  0.989284
2                1.715899  15.492611  12.592684  0.997362

Obserbaciones:

Grupo Etario:

El desempeño es bastante homogéneo. Los mayores tienen el menor MAE y el NDCG más alta, aunque las diferencias son pequeñas.

En cuanto a CG y DCG, los adultos presentan los valores más altos. Sin embargo esto puede deberse a que los grupos tienen distinta cantidad de iteracciones evaluadas y no necesariamente a una mejor calidad de predicción.

Grupo separado por historial:

Los usuario con historial largo presentan un MAE ligeramente mayor que los usuarios con historial corto. Ademas presenta valores superores de CG y DCG lo cual es esperable dado que tiene más libros evaluados.
Lo mismo que se vio en los otros modelo el NDCG en el historial corto es casi perfecto lo cual se tiene que tomar con cautela. 

Grado de exigencia:

Los usuario exigentes presentan nuevamente un MAE mucho mayor, mientras que los normales obtiene el menor error. El patron se mantiene en cada uno de los modelos.

A pesar de esto su NDCG sigue siendo elevado. Esto sugiere que el modelo puede ordenar relativamente bien los ítems, aunque sus predicciones numéricas estén alejadas de las calificaciones reales.




### Test t para grupos separados por historial

In [10]:
from scipy import stats

# Separar los usuarios en dos submuestras según su historial
g_corto = df_evaluacion[df_evaluacion['Grupo_Historial'] == 0]
g_largo = df_evaluacion[df_evaluacion['Grupo_Historial'] == 1]

metricas = ['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']

print("="*60)
print(" PRUEBA T DE STUDENT (WELCH) - HISTORIAL CORTO VS. LARGO")
print("="*60)

for metrica in metricas:
    # Extracción de valores limpios
    v_corto = g_corto[metrica].dropna()
    v_largo = g_largo[metrica].dropna()
    
    # Ejecución del T-test
    t_stat, p_val = stats.ttest_ind(v_corto, v_largo, equal_var=False)
    
    # Interpretación del p-value (umbral alpha = 0.05)
    es_significativo = "SÍ (Diferencia estadísticamente significativa)" if p_val < 0.05 else "NO (Sin evidencia de diferencia)"
    
    print(f"\n[Métrica: {metrica}]")
    print(f"  • Promedio (Historial Corto) : {v_corto.mean():.4f}")
    print(f"  • Promedio (Historial Largo) : {v_largo.mean():.4f}")
    print(f"  • Estadístico t              : {t_stat:.4f}")
    print(f"  • p-value                    : {p_val:.4e}")
    print(f"  • ¿Existe inequidad?         : {es_significativo}")

 PRUEBA T DE STUDENT (WELCH) - HISTORIAL CORTO VS. LARGO

[Métrica: MAE]
  • Promedio (Historial Corto) : 1.3554
  • Promedio (Historial Largo) : 1.3877
  • Estadístico t              : -1.6435
  • p-value                    : 1.0032e-01
  • ¿Existe inequidad?         : NO (Sin evidencia de diferencia)

[Métrica: CG@10]
  • Promedio (Historial Corto) : 7.6592
  • Promedio (Historial Largo) : 17.2927
  • Estadístico t              : -58.6324
  • p-value                    : 0.0000e+00
  • ¿Existe inequidad?         : SÍ (Diferencia estadísticamente significativa)

[Métrica: DCG@10]
  • Promedio (Historial Corto) : 7.6488
  • Promedio (Historial Largo) : 12.3932
  • Estadístico t              : -61.3141
  • p-value                    : 0.0000e+00
  • ¿Existe inequidad?         : SÍ (Diferencia estadísticamente significativa)

[Métrica: NDCG@10]
  • Promedio (Historial Corto) : 0.9999
  • Promedio (Historial Largo) : 0.9870
  • Estadístico t              : 44.1378
  • p-value             

La prueba T no encontró diferencias significativas en MAE entre los usuarios de historial corto y largo. Las diferencias observadas en CG y DCG reflejan principalmente la distinta cantidad de interacciones evaluadas, mientras que NDCG presenta una diferencia estadística pequeña.

### Anova para grupos etarios

In [11]:
from scipy import stats

metricas = ['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']

print("="*60)
print(" TEST ANOVA: GRUPOS ETARIOS")
print("="*60)

for metrica in metricas:
    # 1. Agrupar las muestras por cada rango etario
    grupos = [
        datos[metrica].dropna() 
        for _, datos in df_evaluacion.groupby('Grupo_Etario')
    ]
    
    # 2. ANOVA de un factor
    f_stat, p_val = stats.f_oneway(*grupos)
    
    es_significativo = p_val < 0.05
    conclusion = "SÍ (Diferencias significativas entre edades)" if es_significativo else "NO (Métrica homogénea entre edades)"
    
    print(f"\n[Métrica: {metrica}]")
    print(f"  • Estadístico F    : {f_stat:.4f}")
    print(f"  • p-value          : {p_val:.4e}")
    print(f"  • ¿Existe inequidad?: {conclusion}")

 TEST ANOVA: GRUPOS ETARIOS

[Métrica: MAE]
  • Estadístico F    : 0.3208
  • p-value          : 7.2556e-01
  • ¿Existe inequidad?: NO (Métrica homogénea entre edades)

[Métrica: CG@10]
  • Estadístico F    : 28.2920
  • p-value          : 5.5793e-13
  • ¿Existe inequidad?: SÍ (Diferencias significativas entre edades)

[Métrica: DCG@10]
  • Estadístico F    : 24.3651
  • p-value          : 2.7754e-11
  • ¿Existe inequidad?: SÍ (Diferencias significativas entre edades)

[Métrica: NDCG@10]
  • Estadístico F    : 10.1696
  • p-value          : 3.8704e-05
  • ¿Existe inequidad?: SÍ (Diferencias significativas entre edades)


No detectó diferencias significativas en el error de predicción entre los grupos etarios, lo que indica un desempeño similar del modelo en términos de MAE. Sí se observaron diferencias sigificativas en CG, DCG y NDCG.

### Anova para grupos separados por exigencia

In [12]:
from scipy import stats

metricas = ['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']

print("="*60)
print(" TEST ANOVA - GRADO DE EXIGENCIA")
print("="*60)

for metrica in metricas:
    # 1. Agrupar muestras por grupo de exigencia (0: Exigente, 1: Normal, 2: Generoso)
    grupos = [
        datos[metrica].dropna() 
        for _, datos in df_evaluacion.groupby('Grupo_Exigencia')
    ]
    
    # 2. ANOVA de un factor
    f_stat, p_val = stats.f_oneway(*grupos)
    
    es_significativo = p_val < 0.05
    conclusion = "SÍ (Diferencias significativas según exigencia)" if es_significativo else "NO (Métrica homogénea)"
    
    print(f"\n[Métrica: {metrica}]")
    print(f"  • Estadístico F    : {f_stat:.4f}")
    print(f"  • p-value          : {p_val:.4e}")
    print(f"  • ¿Existe inequidad?: {conclusion}")

 TEST ANOVA - GRADO DE EXIGENCIA

[Métrica: MAE]
  • Estadístico F    : 1439.6182
  • p-value          : 0.0000e+00
  • ¿Existe inequidad?: SÍ (Diferencias significativas según exigencia)

[Métrica: CG@10]
  • Estadístico F    : 194.3328
  • p-value          : 3.9052e-84
  • ¿Existe inequidad?: SÍ (Diferencias significativas según exigencia)

[Métrica: DCG@10]
  • Estadístico F    : 436.5384
  • p-value          : 2.0468e-185
  • ¿Existe inequidad?: SÍ (Diferencias significativas según exigencia)

[Métrica: NDCG@10]
  • Estadístico F    : 58.4454
  • p-value          : 5.1047e-26
  • ¿Existe inequidad?: SÍ (Diferencias significativas según exigencia)


El ANOVA detectó diferencias estadísticamente significativas entre los grupos según el grado de exigencia en todas las métricas. La diferencia más importante se observa en MAE, lo que indica un desempeño desigual en la predicción de calificaciones. 

# Conclusión de KNN ítem-ítem

presenta un raknig muy bueno, con NDCG elevado en todos los grupos. El MAE es similar entre los grupos etarios y entre usuarios con historial corto y largo.

Presenta una diferencia importante según el grado de exigencia: los usuarios exigentes tienen un error de predicción mucho mayor. Esto indeca que el modelo ordena bien los ítems, pero no siempre estima correctamente el valor numérico de sus calificaciones.